In [ ]:
import dataclasses

import jax
import numpy as np

from openpi.models import model as _model
from openpi.policies import pika_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

from zmq_bridge import zmq_image_bridge
from zmq_bridge import zmq_joint_pos_bridge
from zmq_bridge import zmq_joint_cmd_bridge

In [ ]:
joint_cmd_sender = zmq_joint_cmd_bridge.ZMQJointCmdSender("tcp://localhost:6000")
# joint_cmd = np.array([0.0,0.0,0.0,0.0,0.0,0.0,1.0])
joint_cmd = np.array([0.1,0.2,-0.2,0.3,-0.2,0.5,0.01])
joint_cmd_sender.send_array(joint_cmd)

In [ ]:
camera_receiver = zmq_image_bridge.ZMQImageReceiver("tcp://localhost:5555", resize=(224, 224))
fisheye_receiver = zmq_image_bridge.ZMQImageReceiver("tcp://localhost:5556", resize=(224, 224))
joint_pos_receiver = zmq_joint_pos_bridge.ZMQJointPosReceiver("tcp://localhost:5557")
joint_cmd_sender = zmq_joint_cmd_bridge.ZMQJointCmdSender("tcp://localhost:6000")

camera_img = camera_receiver.receive_once()
fisheye_img = fisheye_receiver.receive_once()
joint_pos = joint_pos_receiver.receive_once()
task = "do something"

pika_input = {
    "observation/state": joint_pos,
    "observation/image": camera_img,
    "observation/wrist_image": fisheye_img,
    "task": task,
}

camera_receiver.close()
fisheye_receiver.close()
joint_pos_receiver.close()

# Policy inference

The following example shows how to create a policy from a checkpoint and run inference on a dummy example.

In [ ]:
config = _config.get_config("pi0_pika_lora")
checkpoint_dir = "/home/markov/openpi/checkpoints/pi0_pika_lora/my_experiment_lora/49"

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
# pika_input = pika_policy.make_pika_example()
result = policy.infer(pika_input)

# Delete the policy to free up memory.
del policy

print("Actions shape:", result["actions"].shape)
print(result["actions"][2])

# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [ ]:
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [ ]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)